# Databricks → Azure ML Model Endpoint

Call an Azure ML managed online endpoint from Databricks using Databricks-native capabilities.

## What This Tests

✓ Databricks can call Azure ML model endpoints  
✓ Token-based authentication via Databricks managed identity  
✓ Send inference requests and receive predictions  
✓ No complex Azure SDK dependencies  

## Architecture

```
Databricks Notebook
    ↓ (Databricks Managed Identity)
    ↓ GET token via dbutils.notebook.run() or local token service
    ↓ HTTP POST to Azure ML endpoint
Azure ML Managed Online Endpoint
    ↓ Returns predictions
Databricks → Results
```

## Prerequisites

1. Azure ML endpoint running
2. Credentials in Databricks secret scope `azureml-kv-scope`:
   - `azureml-endpoint-url` - Full endpoint URL
   - `azureml-endpoint-name` - Endpoint name (for reference)

OR set as Spark config:
```
spark.conf.set("azureml.endpoint.url", "https://...")
```

## Note

Uses Databricks' native token environment - no Azure SDK needed for simple endpoint calls!

In [ ]:
import json
import requests
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ Required packages imported")

# Get credentials from Databricks secret scope
secret_scope = "azureml-kv-scope"

try:
    # Try to get endpoint from Databricks secrets
    endpoint_url = dbutils.secrets.get(scope=secret_scope, key="azureml-endpoint-url")
    endpoint_name = dbutils.secrets.get(scope=secret_scope, key="azureml-endpoint-name")
    print("✓ Retrieved Azure ML endpoint from Databricks secrets")
except Exception as e:
    # Fallback to Spark config
    endpoint_url = spark.conf.get("azureml.endpoint.url", None)
    endpoint_name = spark.conf.get("azureml.endpoint.name", "unknown")
    if not endpoint_url:
        print(f"✗ Error: {e}")
        print("\nStore credentials in Databricks secret scope:")
        print(f"  dbutils.secrets.createScope('{secret_scope}')")
        print(f"  dbutils.secrets.put('{secret_scope}', 'azureml-endpoint-url', 'https://...')")
        raise

print(f"Endpoint URL: {endpoint_url[:50]}...")
print(f"Endpoint Name: {endpoint_name}")

In [ ]:
# Get token from Databricks environment
# Databricks injects DATABRICKS_TOKEN in the environment
import os

databricks_token = os.getenv("DATABRICKS_TOKEN")

if not databricks_token:
    print("⚠️  Warning: DATABRICKS_TOKEN not in environment")
    print("   This endpoint likely requires a different token type")
    print("   Try using Azure managed identity or a personal access token")
else:
    print("✓ Databricks token available")
    print("  This can be used for Databricks API calls")
    
print("\nNote: For Azure ML endpoints, you may need:")
print("  - Azure managed identity (automatic in Azure ML compute)")
print("  - Service principal credentials (stored in Databricks secrets)")
print("  - Personal access token for Databricks endpoints")

In [ ]:
# Summary report
summary = {
    "timestamp": datetime.now().isoformat(),
    "test": "Databricks → Azure ML Endpoint Call",
    "status": "passed" if test_passed else "failed",
    "endpoint": endpoint_url[:50] + "...",
    "endpoint_name": endpoint_name,
    "method": "HTTP POST with Databricks environment"
}

print("\n" + "="*70)
print("TEST SUMMARY")
print("="*70)
print(json.dumps(summary, indent=2))
print("="*70)

if test_passed:
    print("\n✅ SUCCESS: Databricks can call Azure ML endpoints!")
    print("   Use this pattern in your production notebooks/jobs")
else:
    print("\n⚠️  TROUBLESHOOTING:")
    print("   1. Verify endpoint URL is correct")
    print("   2. Check Databricks secret scope configuration")
    print("   3. Ensure endpoint requires the authentication method used")
    print("   4. Test endpoint directly from Azure ML UI")


## Summary


In [ ]:
# Make request to Azure ML endpoint
test_passed = False

try:
    print(f"\n🚀 Calling endpoint: {endpoint_url[:60]}...")
    
    response = requests.post(
        endpoint_url,
        json=payload,
        headers=headers,
        timeout=30
    )
    
    print(f"\n📊 Response Status: {response.status_code}")
    
    if response.status_code == 200:
        print("✅ TEST PASSED: Endpoint call successful")
        result = response.json()
        print(f"\n📈 Predictions:")
        print(json.dumps(result, indent=2))
        test_passed = True
        
    elif response.status_code in [401, 403]:
        print("⚠️  Authentication issue - check credentials")
        print(f"   Response: {response.text[:200]}")
        
    else:
        print(f"✗ Endpoint returned status {response.status_code}")
        print(f"   Response: {response.text[:200]}")
        
except requests.exceptions.Timeout:
    print("✗ Request timed out - endpoint may be unavailable")
    
except requests.exceptions.ConnectionError as e:
    print(f"✗ Connection error - check endpoint URL: {e}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# Prepare inference request
# Adjust payload based on your model's input schema

payload = {
    "data": [[1.0, 2.0, 3.0, 4.0]]  # Example numeric features
}

# For JSON input schema:
# payload = {"dataframe_split": {"columns": ["col1", "col2"], "data": [[1, 2], [3, 4]]}}

print("Inference Payload:")
print(json.dumps(payload, indent=2))

# Prepare headers
headers = {
    "Content-Type": "application/json"
}

# If endpoint requires authentication, add token
if databricks_token:
    headers["Authorization"] = f"Bearer {databricks_token}"

print("\nHeaders (token redacted):")
for key, value in headers.items():
    if "Authorization" in key:
        print(f"  {key}: Bearer ****")
    else:
        print(f"  {key}: {value}")


## Call Azure ML Endpoint

Test inference request to the Azure ML model endpoint.
